In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import os
try:
    df = pd.read_csv(f"{path}/Q1_data.csv")
except:
    df = pd.read_csv("Q1_data.csv")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.histplot(df['Delivery_Time'], kde=True)
plt.show()

In [ ]:
if 'Order_ID' in df.columns:
    df = df.drop(columns=['Order_ID'])
    print("Column 'Order_ID' dropped.")
else:
    print("Column 'Order_ID' not found.")

In [ ]:
num_cols = df.select_dtypes(include=['number']).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

cat_cols = df.select_dtypes(include=['object', 'category']).columns
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print(f"Missing values remaining: {df.isnull().sum().sum()}")

In [ ]:
duplicates_count = df.duplicated().sum()
if duplicates_count > 0:
    df.drop_duplicates(inplace=True)

print(f"Removed {duplicates_count} duplicate rows.")

In [ ]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')


encoded_array = encoder.fit_transform(df[categorical_cols])
encoded_df = pd.DataFrame(
    encoded_array,
    columns=encoder.get_feature_names_out(categorical_cols),
    index=df.index
)
df = df.drop(columns=categorical_cols)
df = pd.concat([df, encoded_df], axis=1)

print(f"Encoding complete. New shape: {df.shape}")

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
target_col = 'Delivery_Time'

features = [c for c in df.columns if c != target_col]

df[features] = scaler.fit_transform(df[features])

print("Feature scaling applied.")
df.head()

In [ ]:

#comment: For the models im using RF and CatB a 0.56 is not an issue Tree based are robust to distribution they do not require a perfectly normal target and this is normal life so its considered very good for normal data behaveiour i don't think we need to apply a log transformation or fix this

target_col = 'Delivery_Time'
skewness = df[target_col].skew()
print(f"Target Skewness: {skewness:.4f}")

if abs(skewness) > 1:
    print("Conclusion: The target distribution is HIGHLY skewed.")
elif abs(skewness) > 0.5:
    print("Conclusion: The target distribution is MODERATELY skewed.")
else:
    print("Conclusion: The target distribution is fairly symmetrical.")

In [ ]:
X = df.drop(columns=['Delivery_Time'])
y = df['Delivery_Time']

In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = RandomForestRegressor(random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    scores.append(mean_absolute_error(y_test, y_pred))

print(np.mean(scores))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

importances = pd.Series(model.feature_importances_, index=X.columns)
importances.nlargest(10).plot(kind='barh')
plt.show()

In [ ]:
import seaborn as sns
sns.histplot(y_pred, kde=True)
plt.show()

In [ ]:
#i installed catboost REG casue its not installed with colab
!pip install catboost

In [ ]:
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import numpy as np

X = df.drop(columns=['Delivery_Time'])
y = df['Delivery_Time']

kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores_ensemble = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    rf = RandomForestRegressor(random_state=42)
    cb = CatBoostRegressor(verbose=0, random_state=42)

    rf.fit(X_train, y_train)
    cb.fit(X_train, y_train)


    pred_rf = rf.predict(X_test)
    pred_cb = cb.predict(X_test)


    pred_avg = (pred_rf + pred_cb) / 2


    scores_ensemble.append(mean_absolute_error(y_test, pred_avg))

print(f"Ensemble Average MAE: {np.mean(scores_ensemble)}")